In [1]:
# ============================================================
# Real-Time Cloud-Based Event Planning & RSVP Tracker  (Google Colab, ONE cell)
# Backend : Python + SQLite (transactions, UNIQUE(event_id,user_id), RBAC, waitlist)
# Frontend: live HTML dashboard wired to Python via google.colab output callback
# Real-time: dashboard polls every 2s (same idea as Firestore onSnapshot / WebSocket push)
# Data    : 100% synthetic.  Just run the cell.
# ============================================================
import sqlite3, os, json, time, random, threading
from datetime import datetime, timedelta
from IPython.display import HTML, JSON, display
from google.colab import output

DB = "/tmp/rsvp_tracker.db"
if os.path.exists(DB): os.remove(DB)
now = lambda: datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def conn():
    c = sqlite3.connect(DB, timeout=20, isolation_level=None)
    c.row_factory = sqlite3.Row
    return c

# ---------------- Schema ----------------
with conn() as c:
    c.executescript("""
    PRAGMA journal_mode=WAL;
    CREATE TABLE users(id INTEGER PRIMARY KEY, name TEXT, role TEXT CHECK(role IN('organizer','attendee','admin')));
    CREATE TABLE events(id INTEGER PRIMARY KEY, organizer_id INTEGER REFERENCES users(id), name TEXT, description TEXT,
        venue TEXT, event_date TEXT, capacity INTEGER CHECK(capacity>0), deadline TEXT,
        status TEXT DEFAULT 'PUBLISHED', created_at TEXT);
    CREATE TABLE rsvps(id INTEGER PRIMARY KEY, event_id INTEGER REFERENCES events(id), user_id INTEGER REFERENCES users(id),
        status TEXT CHECK(status IN('GOING','MAYBE','NOT_GOING','WAITLISTED')), wseq REAL,
        responded_at TEXT, updated_at TEXT, UNIQUE(event_id,user_id));
    CREATE INDEX idx_rsvp_event_status ON rsvps(event_id,status);
    CREATE TABLE announcements(id INTEGER PRIMARY KEY, event_id INTEGER, title TEXT, message TEXT, created_at TEXT);
    CREATE TABLE notifications(id INTEGER PRIMARY KEY, user_id INTEGER, event_id INTEGER, type TEXT, message TEXT, created_at TEXT);
    CREATE TABLE audit_logs(id INTEGER PRIMARY KEY, user_id INTEGER, action TEXT, detail TEXT, created_at TEXT);
    """)
    c.execute("INSERT INTO users(name,role) VALUES('Olivia Organizer','organizer')")
    c.execute("INSERT INTO users(name,role) VALUES('Oscar Organizer','organizer')")
    for n in ["Aarav","Bianca","Chen","Diya","Elena","Farid","Grace","Hiro","Isha","Jonas","Kavya","Liam",
              "Maya","Noah","Omar","Priya","Quinn","Rohan","Sara","Tariq"]:
        c.execute("INSERT INTO users(name,role) VALUES(?, 'attendee')", (n + " Attendee",))
    dl = (datetime.now() + timedelta(days=6)).strftime("%Y-%m-%d")
    ed = (datetime.now() + timedelta(days=7)).strftime("%Y-%m-%d")
    c.execute("INSERT INTO events(organizer_id,name,description,venue,event_date,capacity,deadline,created_at) VALUES(1,?,?,?,?,?,?,?)",
              ("Cloud Computing Workshop", "Hands-on serverless + real-time databases.", "Room 101 / Online", ed, 10, dl, now()))

class Err(Exception): pass

def notify(c, uid, eid, typ, msg):
    c.execute("INSERT INTO notifications(user_id,event_id,type,message,created_at) VALUES(?,?,?,?,?)", (uid, eid, typ, msg, now()))

def audit(c, uid, act, detail=""):
    c.execute("INSERT INTO audit_logs(user_id,action,detail,created_at) VALUES(?,?,?,?)", (uid, act, detail, now()))

def user(c, uid):
    u = c.execute("SELECT * FROM users WHERE id=?", (uid,)).fetchone()
    if not u: raise Err("401 Unauthenticated")
    return u

def own_event(c, uid, eid):
    """RBAC: only the organizer who owns the event may manage it."""
    u, e = user(c, uid), c.execute("SELECT * FROM events WHERE id=?", (eid,)).fetchone()
    if not e: raise Err("404 Event not found")
    if u["role"] != "organizer" or e["organizer_id"] != uid: raise Err("403 Forbidden: you do not own this event")
    return e

def refresh_status(c, eid):
    e = c.execute("SELECT * FROM events WHERE id=?", (eid,)).fetchone()
    if e["status"] in ("CANCELLED", "COMPLETED"): return
    g = c.execute("SELECT COUNT(*) FROM rsvps WHERE event_id=? AND status='GOING'", (eid,)).fetchone()[0]
    c.execute("UPDATE events SET status=? WHERE id=?", ("FULL" if g >= e["capacity"] else "PUBLISHED", eid))

# ---------------- RSVP with concurrency-safe capacity ----------------
def rsvp(uid, eid, status):
    if status not in ("GOING", "MAYBE", "NOT_GOING"): raise Err("400 Invalid status")
    c = conn()
    try:
        c.execute("BEGIN IMMEDIATE")            # write lock: check + write become ONE atomic unit (no race)
        u = user(c, uid)
        if u["role"] != "attendee": raise Err("403 Only attendees can RSVP")
        e = c.execute("SELECT * FROM events WHERE id=?", (eid,)).fetchone()
        if not e: raise Err("404 Event not found")
        if e["status"] == "CANCELLED": raise Err("409 Event is cancelled")
        if datetime.now().strftime("%Y-%m-%d") > e["deadline"]: raise Err("409 Registration deadline passed")
        cur = c.execute("SELECT * FROM rsvps WHERE event_id=? AND user_id=?", (eid, uid)).fetchone()
        final = status
        if status == "GOING" and not (cur and cur["status"] == "GOING"):
            going = c.execute("SELECT COUNT(*) FROM rsvps WHERE event_id=? AND status='GOING'", (eid,)).fetchone()[0]
            if going >= e["capacity"]: final = "WAITLISTED"
        was_going = cur and cur["status"] == "GOING"
        keep_seq = cur["wseq"] if cur and cur["status"] == "WAITLISTED" and final == "WAITLISTED" else time.time()
        c.execute("""INSERT INTO rsvps(event_id,user_id,status,wseq,responded_at,updated_at) VALUES(?,?,?,?,?,?)
                     ON CONFLICT(event_id,user_id) DO UPDATE SET status=excluded.status,wseq=?,updated_at=excluded.updated_at""",
                  (eid, uid, final, keep_seq, now(), now(), keep_seq))
        msg = "Your RSVP has been recorded: %s" % final.replace("_", " ")
        if final == "WAITLISTED": msg = "Event is full - you joined the waitlist."
        notify(c, uid, eid, "RSVP", msg)
        if was_going and final != "GOING":          # FIFO promotion
            nxt = c.execute("SELECT * FROM rsvps WHERE event_id=? AND status='WAITLISTED' ORDER BY wseq LIMIT 1", (eid,)).fetchone()
            if nxt:
                c.execute("UPDATE rsvps SET status='GOING',updated_at=? WHERE id=?", (now(), nxt["id"]))
                notify(c, nxt["user_id"], eid, "WAITLIST", "Good news! You were promoted from the waitlist to GOING.")
        refresh_status(c, eid)
        audit(c, uid, "RSVP", "event=%s status=%s" % (eid, final))
        c.execute("COMMIT")
        return final
    except Exception:
        try: c.execute("ROLLBACK")
        except Exception: pass
        raise
    finally:
        c.close()

def cancel_rsvp(uid, eid):
    c = conn()
    try:
        r = c.execute("SELECT status FROM rsvps WHERE event_id=? AND user_id=?", (eid, uid)).fetchone()
        if not r: raise Err("404 No RSVP to cancel")
    finally: c.close()
    return rsvp(uid, eid, "NOT_GOING")

# ---------------- Read model ----------------
def state(uid, eid):
    c = conn()
    try:
        u = user(c, uid)
        evs = []
        for e in c.execute("SELECT * FROM events ORDER BY id DESC"):
            cnt = {r[0]: r[1] for r in c.execute("SELECT status,COUNT(*) FROM rsvps WHERE event_id=? GROUP BY status", (e["id"],))}
            evs.append(dict(e, going=cnt.get("GOING", 0)))
        out = {"me": dict(u), "events": evs, "users": [dict(x) for x in c.execute("SELECT * FROM users")]}
        e = c.execute("SELECT * FROM events WHERE id=?", (eid,)).fetchone()
        if e:
            cnt = {r[0]: r[1] for r in c.execute("SELECT status,COUNT(*) FROM rsvps WHERE event_id=? GROUP BY status", (eid,))}
            total_att = c.execute("SELECT COUNT(*) FROM users WHERE role='attendee'").fetchone()[0]
            resp = sum(cnt.values())
            out["event"] = dict(e)
            out["stats"] = {"going": cnt.get("GOING", 0), "maybe": cnt.get("MAYBE", 0), "not_going": cnt.get("NOT_GOING", 0),
                            "waitlist": cnt.get("WAITLISTED", 0), "responded": resp, "invited": total_att,
                            "response_rate": round(resp / total_att * 100, 1) if total_att else 0,
                            "available": max(0, e["capacity"] - cnt.get("GOING", 0)),
                            "utilization": round(cnt.get("GOING", 0) / e["capacity"] * 100, 1),
                            "conversion": round(cnt.get("GOING", 0) / resp * 100, 1) if resp else 0}
            out["timeline"] = [r[0] for r in c.execute("SELECT status FROM rsvps WHERE event_id=? ORDER BY updated_at,id", (eid,))]
            out["announcements"] = [dict(a) for a in c.execute("SELECT * FROM announcements WHERE event_id=? ORDER BY id DESC", (eid,))]
            m = c.execute("SELECT status FROM rsvps WHERE event_id=? AND user_id=?", (eid, uid)).fetchone()
            out["my_rsvp"] = m[0] if m else None
            if u["role"] == "organizer" and e["organizer_id"] == uid:
                out["attendees"] = [dict(r) for r in c.execute(
                    "SELECT u.name,r.status,r.updated_at FROM rsvps r JOIN users u ON u.id=r.user_id WHERE r.event_id=? ORDER BY r.updated_at DESC", (eid,))]
        out["notifications"] = [dict(n) for n in c.execute("SELECT * FROM notifications WHERE user_id=? ORDER BY id DESC LIMIT 15", (uid,))]
        out["audit"] = [dict(a) for a in c.execute("SELECT * FROM audit_logs ORDER BY id DESC LIMIT 8")]
        return out
    finally: c.close()

# ---------------- Organizer actions ----------------
def create_event(uid, p):
    c = conn()
    try:
        u = user(c, uid)
        if u["role"] != "organizer": raise Err("403 Only organizers can create events")
        cap = int(p.get("capacity", 0))
        if cap <= 0: raise Err("400 Capacity must be positive")
        datetime.strptime(p["date"], "%Y-%m-%d"); datetime.strptime(p["deadline"], "%Y-%m-%d")
        if p["deadline"] > p["date"]: raise Err("400 Deadline must be on/before event date")
        if not p["name"].strip(): raise Err("400 Name required")
        cur = c.execute("INSERT INTO events(organizer_id,name,description,venue,event_date,capacity,deadline,created_at) VALUES(?,?,?,?,?,?,?,?)",
                        (uid, p["name"].strip(), p.get("desc", ""), p.get("venue", "Online"), p["date"], cap, p["deadline"], now()))
        audit(c, uid, "CREATE_EVENT", p["name"])
        return cur.lastrowid
    except ValueError: raise Err("400 Invalid date (use YYYY-MM-DD)")
    finally: c.close()

def announce(uid, eid, title, msg):
    c = conn()
    try:
        own_event(c, uid, eid)
        if not title.strip() or not msg.strip(): raise Err("400 Title and message required")
        c.execute("INSERT INTO announcements(event_id,title,message,created_at) VALUES(?,?,?,?)", (eid, title, msg, now()))
        for r in c.execute("SELECT user_id FROM rsvps WHERE event_id=? AND status!='NOT_GOING'", (eid,)).fetchall():
            notify(c, r[0], eid, "ANNOUNCEMENT", "%s: %s" % (title, msg))
        audit(c, uid, "ANNOUNCE", title)
    finally: c.close()

def cancel_event(uid, eid):
    c = conn()
    try:
        own_event(c, uid, eid)
        c.execute("UPDATE events SET status='CANCELLED' WHERE id=?", (eid,))
        for r in c.execute("SELECT user_id FROM rsvps WHERE event_id=?", (eid,)).fetchall():
            notify(c, r[0], eid, "CANCELLED", "The event has been cancelled by the organizer.")
        audit(c, uid, "CANCEL_EVENT", str(eid))
    finally: c.close()

def simulate(uid, eid, n):
    own_event(conn(), uid, eid)
    ids = [r[0] for r in conn().execute("SELECT id FROM users WHERE role='attendee'")]
    for i in random.sample(ids, min(n, len(ids))):
        try: rsvp(i, eid, random.choice(["GOING", "GOING", "MAYBE", "NOT_GOING"]))
        except Err: pass

# ---------------- Built-in test suite ----------------
def run_tests(eid):
    log = []
    def t(name, ok, info=""): log.append({"name": name, "pass": bool(ok), "info": info})
    ids = [r[0] for r in conn().execute("SELECT id FROM users WHERE role='attendee'")]
    # T1 race condition: create event with 1 seat, 15 simultaneous GOING requests
    c = conn(); cur = c.execute("INSERT INTO events(organizer_id,name,description,venue,event_date,capacity,deadline,created_at) VALUES(1,'Race Test (1 seat)','','',?,1,?,?)",
        ((datetime.now()+timedelta(days=3)).strftime("%Y-%m-%d"), (datetime.now()+timedelta(days=2)).strftime("%Y-%m-%d"), now())); rid = cur.lastrowid; c.close()
    res = []
    th = [threading.Thread(target=lambda i=i: res.append(rsvp(i, rid, "GOING"))) for i in ids[:15]]
    [x.start() for x in th]; [x.join() for x in th]
    t("Concurrency: 15 users fight for last seat", res.count("GOING") == 1, "accepted=%d waitlisted=%d" % (res.count("GOING"), res.count("WAITLISTED")))
    # T2 duplicate prevention
    rsvp(ids[0], eid, "MAYBE"); rsvp(ids[0], eid, "MAYBE")
    n = conn().execute("SELECT COUNT(*) FROM rsvps WHERE event_id=? AND user_id=?", (eid, ids[0])).fetchone()[0]
    t("Duplicate RSVP prevented (UNIQUE event,user)", n == 1, "rows=%d" % n)
    # T3 update MAYBE->GOING
    t("Update MAYBE -> GOING", rsvp(ids[0], eid, "GOING") in ("GOING", "WAITLISTED"))
    # T4 unauthorized
    for nm, fn in [("Attendee cannot create event", lambda: create_event(ids[0], {"name": "x", "capacity": 5, "date": "2030-01-02", "deadline": "2030-01-01"})),
                   ("Other organizer cannot cancel event", lambda: cancel_event(2, eid)),
                   ("Attendee cannot post announcement", lambda: announce(ids[0], eid, "a", "b"))]:
        try: fn(); t(nm, False, "no error raised")
        except Err as ex: t(nm, str(ex).startswith("403"), str(ex))
    # T5 waitlist FIFO promotion
    wid = conn().execute("SELECT user_id FROM rsvps WHERE event_id=? AND status='WAITLISTED' ORDER BY wseq LIMIT 1", (rid,)).fetchone()
    g = conn().execute("SELECT user_id FROM rsvps WHERE event_id=? AND status='GOING'", (rid,)).fetchone()[0]
    rsvp(g, rid, "NOT_GOING")
    promoted = conn().execute("SELECT status FROM rsvps WHERE event_id=? AND user_id=?", (rid, wid[0])).fetchone()[0]
    t("Waitlist FIFO promotion on cancellation", promoted == "GOING", "first waitlisted -> %s" % promoted)
    # T6 validation
    try: create_event(1, {"name": "Bad", "capacity": 0, "date": "2030-01-02", "deadline": "2030-01-01"}); t("Invalid capacity rejected", False)
    except Err as ex: t("Invalid capacity rejected", str(ex).startswith("400"), str(ex))
    conn().execute("DELETE FROM events WHERE id=?", (rid,)) if False else None
    return log

# ---------------- API bridge (JS -> Python) ----------------
def api(req):
    try:
        req = json.loads(req); a, uid, eid = req["a"], int(req["uid"]), int(req.get("eid") or 0)
        if a == "rsvp": return JSON({"ok": True, "status": rsvp(uid, eid, req["status"])})
        if a == "cancel_rsvp": cancel_rsvp(uid, eid)
        elif a == "create_event": return JSON({"ok": True, "id": create_event(uid, req)})
        elif a == "announce": announce(uid, eid, req["title"], req["message"])
        elif a == "cancel_event": cancel_event(uid, eid)
        elif a == "simulate": simulate(uid, eid, int(req.get("n", 5)))
        elif a == "tests": return JSON({"ok": True, "tests": run_tests(eid)})
        return JSON({"ok": True, "state": state(uid, eid)})
    except Err as e: return JSON({"ok": False, "error": str(e)})
    except Exception as e: return JSON({"ok": False, "error": "500 " + repr(e)})

output.register_callback("rsvp_api", api)

# ---------------- Dashboard ----------------
HTML_UI = r"""
<style>
:root{--bg:#0b1020;--card:#151c33;--bd:#263056;--tx:#e8ecf8;--mu:#8b96b8;--ac:#6c8cff;--g:#2ecc9a;--y:#f5b942;--r:#ff6b7a;--w:#b37bff}
*{box-sizing:border-box}#app{font-family:Inter,system-ui,sans-serif;background:var(--bg);color:var(--tx);padding:16px;border-radius:14px;min-width:760px}
.top{display:flex;gap:12px;align-items:center;flex-wrap:wrap;margin-bottom:14px}.logo{font-weight:700;font-size:18px}.logo span{color:var(--ac)}
.live{font-size:12px;color:var(--g)}.live:before{content:"";display:inline-block;width:8px;height:8px;border-radius:50%;background:var(--g);margin-right:6px;animation:p 1.2s infinite}
@keyframes p{50%{opacity:.25}}
select,input,textarea,button{background:#0f1529;color:var(--tx);border:1px solid var(--bd);border-radius:8px;padding:8px 10px;font:inherit;font-size:13px}
button{cursor:pointer;background:var(--ac);border:0;font-weight:600}button:hover{filter:brightness(1.1)}button.s{background:#222c52}button.d{background:var(--r)}
button.go{background:var(--g);color:#06281d}button.mb{background:var(--y);color:#3a2800}button.ng{background:var(--r)}
.tabs{display:flex;gap:6px;margin-left:auto}.tabs button{background:#1a2342}.tabs button.on{background:var(--ac)}
.grid{display:grid;gap:12px}.kpis{grid-template-columns:repeat(auto-fit,minmax(130px,1fr));margin-bottom:12px}
.card{background:var(--card);border:1px solid var(--bd);border-radius:12px;padding:14px}.card h3{margin:0 0 10px;font-size:13px;color:var(--mu);font-weight:600;text-transform:uppercase;letter-spacing:.5px}
.k .v{font-size:28px;font-weight:700}.k .l{font-size:11px;color:var(--mu)}
.two{grid-template-columns:1fr 1fr}.three{grid-template-columns:1fr 1fr 1fr}
.bar{height:14px;background:#0f1529;border-radius:8px;overflow:hidden}.bar i{display:block;height:100%;background:linear-gradient(90deg,var(--ac),var(--g));transition:width .5s}
table{width:100%;border-collapse:collapse;font-size:13px}td,th{text-align:left;padding:6px;border-bottom:1px solid var(--bd)}th{color:var(--mu);font-weight:600}
.pill{padding:2px 8px;border-radius:10px;font-size:11px;font-weight:700}.GOING{background:#12392f;color:var(--g)}.MAYBE{background:#3d3010;color:var(--y)}
.NOT_GOING{background:#3c1a20;color:var(--r)}.WAITLISTED{background:#2f2050;color:var(--w)}.PUBLISHED{background:#14294d;color:var(--ac)}.FULL{background:#3d3010;color:var(--y)}.CANCELLED{background:#3c1a20;color:var(--r)}
.row{display:flex;gap:8px;flex-wrap:wrap;margin:8px 0}.row>*{flex:1;min-width:120px}.note{font-size:12px;color:var(--mu)}
#toast{position:fixed;right:18px;bottom:18px;background:#222c52;border:1px solid var(--bd);padding:10px 14px;border-radius:10px;display:none;z-index:9;max-width:340px}
.item{padding:8px;border-bottom:1px solid var(--bd);font-size:13px}.item small{color:var(--mu);display:block}
.ok{color:var(--g)}.bad{color:var(--r)}.hide{display:none}
</style>
<div id="app">
 <div class="top">
  <div class="logo">&#9729; Event<span>Sync</span> <span style="font-size:12px;color:var(--mu);font-weight:400">Real-Time RSVP Tracker</span></div>
  <span class="live" id="live">LIVE</span>
  <label class="note">Acting as</label><select id="who"></select>
  <label class="note">Event</label><select id="ev"></select>
  <div class="tabs"><button id="t1" class="on" onclick="tab(1)">Dashboard</button><button id="t2" onclick="tab(2)">RSVP / Attendee</button><button id="t3" onclick="tab(3)">Manage</button><button id="t4" onclick="tab(4)">Tests &amp; Logs</button></div>
 </div>
 <div id="v1">
  <div id="evhead" style="margin-bottom:10px"></div>
  <div class="grid kpis" id="kpis"></div>
  <div class="grid two" style="margin-bottom:12px">
   <div class="card"><h3>RSVP Distribution</h3><div id="donut"></div></div>
   <div class="card"><h3>Capacity Utilization</h3><div id="cap"></div><h3 style="margin-top:18px">RSVP Growth (cumulative Going)</h3><div id="line"></div></div>
  </div>
  <div class="card"><h3>Attendee Responses (organizer only)</h3><div id="att"></div></div>
 </div>
 <div id="v2" class="hide">
  <div class="grid two">
   <div class="card"><h3>Your RSVP</h3><div id="mine"></div><div class="row"><button class="go" onclick="rs('GOING')">Going</button><button class="mb" onclick="rs('MAYBE')">Maybe</button><button class="ng" onclick="rs('NOT_GOING')">Not Going</button></div>
     <p class="note">Select an <b>attendee</b> in "Acting as". Counts are computed by the backend - the client cannot change them.</p><h3 style="margin-top:14px">Announcements</h3><div id="ann"></div></div>
   <div class="card"><h3>Notifications</h3><div id="notes"></div></div>
  </div>
 </div>
 <div id="v3" class="hide">
  <div class="grid two">
   <div class="card"><h3>Create Event (organizer)</h3>
    <div class="row"><input id="n" placeholder="Event name" value="Serverless Bootcamp"></div>
    <div class="row"><input id="vn" placeholder="Venue" value="Auditorium A"><input id="cp" type="number" placeholder="Capacity" value="25"></div>
    <div class="row"><input id="dt" placeholder="Date YYYY-MM-DD"><input id="dd" placeholder="Deadline YYYY-MM-DD"></div>
    <div class="row"><textarea id="ds" placeholder="Description" rows="2">Synthetic demo event</textarea></div><button onclick="mk()">Create Event</button></div>
   <div class="card"><h3>Organizer Tools</h3>
    <div class="row"><input id="at" placeholder="Announcement title" value="Venue Updated"></div>
    <div class="row"><textarea id="am" rows="2" placeholder="Message">Doors open at 9:30 AM. Bring your laptop.</textarea></div>
    <div class="row"><button onclick="an()">Publish Announcement</button><button class="d" onclick="ce()">Cancel Event</button></div>
    <h3 style="margin-top:14px">Simulate live traffic</h3><div class="row"><button class="s" onclick="sim(1)">+1 random RSVP</button><button class="s" onclick="sim(5)">+5 random RSVPs</button><button class="s" onclick="sim(12)">+12 RSVPs</button></div>
    <p class="note">Open the Dashboard tab and watch counters move without refreshing.</p></div>
  </div>
 </div>
 <div id="v4" class="hide">
  <div class="grid two">
   <div class="card"><h3>Automated Tests</h3><button onclick="rt()">Run test suite (incl. 15-thread race test)</button><div id="tests" style="margin-top:10px"></div></div>
   <div class="card"><h3>Audit Log</h3><div id="audit"></div></div>
  </div>
 </div>
</div><div id="toast"></div>
<script>
let S=null,V=1;const $=id=>document.getElementById(id);
const esc=s=>String(s??"").replace(/[&<>"']/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
const pill=s=>`<span class="pill ${s}">${esc(String(s).replace("_"," "))}</span>`;
function toast(m,bad){const t=$("toast");t.textContent=m;t.style.display="block";t.style.borderColor=bad?"#ff6b7a":"#2ecc9a";clearTimeout(t._h);t._h=setTimeout(()=>t.style.display="none",3200)}
async function call(a,p={}){const r=await google.colab.kernel.invokeFunction("rsvp_api",[JSON.stringify({a,uid:$("who").value||1,eid:$("ev").value||0,...p})],{});return r.data["application/json"]}
async function act(a,p,msg){const r=await call(a,p);if(!r.ok)toast(r.error,1);else{if(msg)toast(msg);if(r.state){S=r.state;draw()}}return r}
function tab(n){V=n;for(let i=1;i<=4;i++){$("v"+i).classList.toggle("hide",i!=n);$("t"+i).classList.toggle("on",i==n)}}
async function poll(){try{const r=await call("state");if(r.ok){S=r.state;draw();$("live").textContent="LIVE \u2022 "+new Date().toLocaleTimeString()}}catch(e){$("live").textContent="reconnecting..."}}
function fillSel(){const w=$("who"),e=$("ev");const cw=w.value,ce=e.value;
 if(!w.options.length||w.options.length!=S.users.length)w.innerHTML=S.users.map(u=>`<option value="${u.id}">${esc(u.name)} (${u.role})</option>`).join("");
 e.innerHTML=S.events.map(x=>`<option value="${x.id}">${esc(x.name)}</option>`).join("");if(cw)w.value=cw;if(ce)e.value=ce}
function draw(){fillSel();const s=S.stats,e=S.event;if(!e)return;
 $("evhead").innerHTML=`<b style="font-size:18px">${esc(e.name)}</b> ${pill(e.status)} <span class="note">&nbsp;${esc(e.venue)} &bull; ${esc(e.event_date)} &bull; RSVP by ${esc(e.deadline)}</span><div class="note">${esc(e.description)}</div>`;
 const K=[["Going",s.going,"var(--g)"],["Maybe",s.maybe,"var(--y)"],["Not Going",s.not_going,"var(--r)"],["Waitlist",s.waitlist,"var(--w)"],["Seats Left",s.available+" / "+e.capacity,"var(--ac)"],["Response Rate",s.response_rate+"%","var(--tx)"],["Conversion",s.conversion+"%","var(--tx)"]];
 $("kpis").innerHTML=K.map(k=>`<div class="card k"><div class="v" style="color:${k[2]}">${k[1]}</div><div class="l">${k[0]}</div></div>`).join("");
 const D=[[s.going,"#2ecc9a","Going"],[s.maybe,"#f5b942","Maybe"],[s.not_going,"#ff6b7a","Not Going"],[s.waitlist,"#b37bff","Waitlist"]],T=D.reduce((a,b)=>a+b[0],0)||1;let off=0;
 $("donut").innerHTML=`<svg viewBox="0 0 200 110" width="100%" height="150"><g transform="translate(55,55) rotate(-90)">${D.map(d=>{const l=d[0]/T*251.3,c=`<circle r="40" fill="none" stroke="${d[1]}" stroke-width="18" stroke-dasharray="${l} 251.3" stroke-dashoffset="${-off}"/>`;off+=l;return c}).join("")}</g><text x="55" y="60" text-anchor="middle" fill="#e8ecf8" font-size="16" font-weight="700">${s.responded}</text>${D.map((d,i)=>`<rect x="115" y="${18+i*20}" width="10" height="10" fill="${d[1]}"/><text x="131" y="${27+i*20}" fill="#e8ecf8" font-size="11">${d[2]}: ${d[0]}</text>`).join("")}</svg>`;
 $("cap").innerHTML=`<div class="bar"><i style="width:${Math.min(100,s.utilization)}%"></i></div><div class="note" style="margin-top:6px">${s.going}/${e.capacity} seats (${s.utilization}%) &bull; ${s.invited} invited, ${s.responded} responded</div>`;
 let c=0;const pts=[0].concat(S.timeline.map(x=>(x=="GOING"&&c++,c)));const mx=Math.max(1,...pts);
 $("line").innerHTML=`<svg viewBox="0 0 300 90" width="100%" height="110"><polyline fill="none" stroke="#6c8cff" stroke-width="2.5" points="${pts.map((p,i)=>(i/Math.max(1,pts.length-1)*290+5)+","+(80-p/mx*70)).join(" ")}"/></svg>`;
 $("att").innerHTML=S.attendees?`<table><tr><th>Attendee</th><th>Status</th><th>Updated</th></tr>${S.attendees.map(a=>`<tr><td>${esc(a.name)}</td><td>${pill(a.status)}</td><td class="note">${esc(a.updated_at)}</td></tr>`).join("")||'<tr><td colspan=3 class="note">No responses yet</td></tr>'}</table>`:'<div class="note">Switch "Acting as" to the event\'s organizer (Olivia) to see attendee details. (RBAC)</div>';
 $("mine").innerHTML=S.my_rsvp?`Current: ${pill(S.my_rsvp)}`:'<span class="note">No RSVP yet</span>';
 $("ann").innerHTML=S.announcements.map(a=>`<div class="item"><b>${esc(a.title)}</b> ${esc(a.message)}<small>${a.created_at}</small></div>`).join("")||'<div class="note">None</div>';
 $("notes").innerHTML=S.notifications.map(n=>`<div class="item">${pill(n.type=="RSVP"?"PUBLISHED":n.type=="WAITLIST"?"WAITLISTED":n.type=="CANCELLED"?"CANCELLED":"MAYBE")} ${esc(n.message)}<small>${n.created_at}</small></div>`).join("")||'<div class="note">No notifications</div>';
 $("audit").innerHTML=S.audit.map(a=>`<div class="item"><b>${esc(a.action)}</b> ${esc(a.detail)}<small>user ${a.user_id} &bull; ${a.created_at}</small></div>`).join("")}
const rs=st=>act("rsvp",{status:st},"RSVP sent");
const mk=()=>act("create_event",{name:$("n").value,venue:$("vn").value,capacity:$("cp").value,date:$("dt").value,deadline:$("dd").value,desc:$("ds").value},"Event created");
const an=()=>act("announce",{title:$("at").value,message:$("am").value},"Announcement published");
const ce=()=>confirm("Cancel this event?")&&act("cancel_event",{},"Event cancelled");
const sim=n=>act("simulate",{n},"Simulated "+n+" RSVPs");
async function rt(){$("tests").innerHTML="Running...";const r=await call("tests");if(!r.ok){$("tests").innerHTML=esc(r.error);return}
 $("tests").innerHTML=r.tests.map(t=>`<div class="item"><span class="${t.pass?"ok":"bad"}">${t.pass?"PASS":"FAIL"}</span> ${esc(t.name)}<small>${esc(t.info)}</small></div>`).join("");poll()}
$("who").onchange=poll;$("ev").onchange=poll;
const d=x=>new Date(Date.now()+x*864e5).toISOString().slice(0,10);$("dt").value=d(14);$("dd").value=d(10);
poll();setInterval(poll,2000);
</script>
"""
display(HTML(HTML_UI))
print("Ready. Tip: use 'Manage' > simulate RSVPs, then watch 'Dashboard'. Switch 'Acting as' to test roles.")

Ready. Tip: use 'Manage' > simulate RSVPs, then watch 'Dashboard'. Switch 'Acting as' to test roles.
